In [1]:
import ast
import json
from collections import Counter
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns

/opt/saturncloud/envs/saturn/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.1
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


## Read in Data (csv has omdb data)

In [2]:
movies = pd.read_csv(
    "eng_movies_gt_2000_with_omdb_dataset_102023.csv", low_memory=False
)
movies_cleaned = pd.read_csv(
    "eng_movies_gt_2000_with_omdb_dataset_102023.csv", low_memory=False
)

In [3]:
missing_movies = movies.isna()  # or df.isnull()
missing_movies.sum()

id                            0
adult                         0
backdrop_path            177362
belongs_to_collection    247368
budget                        0
genres                        0
homepage                 193044
id.1                          0
imdb_id                   84558
original_language             0
original_title                4
overview                      0
popularity                    0
poster_path               62905
production_companies          0
production_countries          0
release_date                  0
revenue                       0
runtime                       0
spoken_languages              0
status                        0
tagline                  195252
title                         4
video                         0
vote_average                  0
vote_count                    0
keywords                      0
omdb_Title                74724
omdb_Year                 74724
omdb_Rated               185781
omdb_Released             97483
omdb_Run

### Clean Genre

* Find out how many rows with genre are empty (i.e. [], Null, None, NaN, etc.)
* Replace [] with NaN values so we can universally replace them if possible
* Change the format of tmdb "genres" column to match omdb "genres" column
* Replace the empty tmdb genres with omdb_Genre
* This saves us around 44,427 Movies which previously had empty genres

### Clean Keywords
* Replace [] with NaN values 
* change format from list of dictionaries to string separated by comma


In [4]:
# Find rows where genre is null/NaN/[] etc.
print(" Number of rows with []: " + str(len(movies[movies["genres"] == "[]"])))
print(" Number of rows with NaN: " + str(len(movies[movies["genres"] == np.nan])))
print(" Number of rows with None: " + str(len(movies[movies["genres"] == None])))
print(" Number of rows with '': " + str(len(movies[movies["genres"] == ""])))

 Number of rows with []: 79723
 Number of rows with NaN: 0
 Number of rows with None: 0
 Number of rows with '': 0


In [5]:
movies_cleaned["genres"] = movies_cleaned["genres"].apply(ast.literal_eval)
movies_cleaned["keywords"] = movies_cleaned["keywords"].apply(ast.literal_eval)

In [6]:
movies_cleaned["genres"][3]

[{'id': 16, 'name': 'Animation'}, {'id': 10751, 'name': 'Family'}]

In [7]:
# movies_cleaned['genres'] = movies_cleaned['genres'].replace([], np.NaN)
movies_cleaned["genres"] = movies_cleaned["genres"].apply(
    lambda y: np.nan if len(y) == 0 else y
)

In [8]:
movies_cleaned["genres"]

0                                                       NaN
1                         [{'id': 10751, 'name': 'Family'}]
2                       [{'id': 99, 'name': 'Documentary'}]
3         [{'id': 16, 'name': 'Animation'}, {'id': 10751...
4         [{'id': 18, 'name': 'Drama'}, {'id': 80, 'name...
                                ...                        
252745                                                  NaN
252746                                                  NaN
252747                                                  NaN
252748                  [{'id': 99, 'name': 'Documentary'}]
252749                  [{'id': 99, 'name': 'Documentary'}]
Name: genres, Length: 252750, dtype: object

In [9]:
# movies_cleaned['keywords'] = movies_cleaned['keywords'].replace('[]', np.NaN)
movies_cleaned["keywords"] = movies_cleaned["keywords"].apply(
    lambda y: np.nan if len(y) == 0 else y
)

In [10]:
type(movies_cleaned["keywords"][0])

list

In [11]:
def convert_list_of_dict_to_str(l):
    s = ""
    if type(l) is list:
        for d in l:
            if "name" in d:
                s += d["name"] + ", "
        return s[:-2]
    else:
        return np.nan

In [12]:
movies_cleaned["genres"] = movies_cleaned["genres"].map(
    lambda x: convert_list_of_dict_to_str(x)
)

In [13]:
movies_cleaned["genres"].isna().sum()

79723

In [14]:
movies_cleaned.genres = movies_cleaned["genres"].fillna(movies_cleaned["omdb_Genre"])

In [15]:
movies_cleaned["genres"].isna().sum()

35296

In [16]:
movies_cleaned["keywords"] = movies_cleaned["keywords"].map(
    lambda x: convert_list_of_dict_to_str(x)
)

In [17]:
movies_cleaned["keywords"].isna().sum()

175080

In [18]:
# OMDB Saved about 79,723 - 35,296 = 44,427
movies_cleaned.isna().sum()

id                            0
adult                         0
backdrop_path            177362
belongs_to_collection    247368
budget                        0
genres                    35296
homepage                 193044
id.1                          0
imdb_id                   84558
original_language             0
original_title                4
overview                      0
popularity                    0
poster_path               62905
production_companies          0
production_countries          0
release_date                  0
revenue                       0
runtime                       0
spoken_languages              0
status                        0
tagline                  195252
title                         4
video                         0
vote_average                  0
vote_count                    0
keywords                 175080
omdb_Title                74724
omdb_Year                 74724
omdb_Rated               185781
omdb_Released             97483
omdb_Run

In [19]:
wikipedia_movies = pd.read_csv(
    "Scraped movie data with preprocesing.csv", low_memory=False
)

In [20]:
wikipedia_movies

,year,href,title,plot,premise,synopsis,directed_by,screenplay_by,based_on,starring,...,num_parentheticals,plot_parenthetical_fuzzy_scores,new_plot,all_writers_first_movie,any_writers_first_movie,roi,roi_binary,roi_multi,rt_binary,rt_multi
0,1973,40_Carats_(film),40 Carats (film),"Ann Stanley, who sells real estate in New York...",NaN,NaN,Directed byMilton Katselas,Screenplay byLeonard Gershe,Based on\nQuarante caratsby Pierre BarilletJea...,Starring\nLiv Ullmann\nEdward Albert\nGene Kel...,...,0,NaN,Forty year old Norwegian-American divorcée Ann...,False,True,NaN,0,0,0,0
1,1973,Ace_Eli_and_Rodger_of_the_Skies,Ace Eli and Rodger of the Skies,"In the early 1920s, Eli (Cliff Robertson) is a...",NaN,NaN,Directed byJohn Erman,Screenplay byClaudia Salter,NaN,Starring\nCliff Robertson\nEric Shea\nPamela F...,...,5,"[('Cliff Robertson', 100), ('Pamela Franklin',...","In the early 1920s, Eli is a barnstorming stu...",False,True,NaN,0,0,0,0
2,1973,The_Affair_(1973_film),The Affair (1973 film),Courtney Patterson is a beautiful 32 year old ...,NaN,NaN,Directed byGilbert Cates,NaN,NaN,StarringNatalie WoodRobert WagnerBruce Davison...,...,4,"[('Davison', 67)]",Courtney Patterson is a beautiful 32 year old ...,True,True,NaN,0,0,0,0
3,1973,The_All-American_Boy_(film),The All-American Boy (film),"Vic Bealer, a young boxer from a small town in...",NaN,NaN,Directed byCharles Eastman,NaN,NaN,StarringJon VoightE. J. Peaker,...,0,NaN,"Vic Bealer, a young boxer from a small town in...",True,True,NaN,0,0,0,0
4,1973,American_Graffiti,American Graffiti,On their last evening of summer vacation in 19...,NaN,NaN,Directed byGeorge Lucas,NaN,NaN,Starring\nRichard Dreyfuss\nRon Howard\nPaul L...,...,0,NaN,On their last evening of summer vacation in 19...,False,True,180.18018,1,2,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12544,2023,Rebel_Moon,Rebel Moon,NaN,In a universe controlled by the corrupt govern...,NaN,Directed byZack Snyder,Screenplay by\nZack Snyder\nKurt Johnstad\nSha...,NaN,Starring\nSofia Boutella\nCharlie Hunnam\nMich...,...,0,NaN,In a universe controlled by the corrupt govern...,False,False,NaN,0,0,0,0
12545,2023,The_Iron_Claw_(film),The Iron Claw (film),NaN,NaN,The film is centered around the Von Erich fami...,Directed bySean Durkin,NaN,NaN,Starring\nZac Efron\nJeremy Allen White\nHarri...,...,0,NaN,The film is centered around the Von Erich fami...,False,False,NaN,0,0,0,0
12546,2023,The_Color_Purple_(2023_film),The Color Purple (2023 film),NaN,A story of the life-long struggles of an Afric...,NaN,Directed byBlitz Bazawule,Screenplay byMarcus Gardley,Based on\nThe Color Purpleby Alice Walker\nThe...,Starring\nTaraji P. Henson\nDanielle Brooks\nC...,...,0,NaN,"Celie is a young poor, uneducated 14-year-old ...",False,True,NaN,0,0,0,0
12547,2023,The_Boys_in_the_Boat_(film),The Boys in the Boat (film),The non-fiction novel describes the University...,NaN,NaN,Directed byGeorge Clooney,Screenplay byMark L. Smith,Based onThe Boys in the Boatby Daniel James Brown,Starring\nCallum Turner\nJoel Edgerton\n,...,0,NaN,The non-fiction novel describes the University...,False,False,NaN,0,0,0,0


In [21]:
wikipedia_movies.shape

(12549, 48)

In [22]:
wikipedia_movies.isna().sum()

year                                   0
href                                   0
title                                  0
plot                                1715
premise                            12333
synopsis                           12204
directed_by                          300
screenplay_by                       8284
based_on                            8337
starring                             517
box_office                          3538
written_by                          4805
budget                              4719
omdb_response                          0
omdb_title                          1570
genres                                 0
omdb_plot                           1720
release_date                        1720
omdb_director                        139
omdb_writer                          323
omdb_actors                           55
imdb_score                          1813
imdb_votes                          1778
rotten_tomatoes_score               4183
metacritic_score

In [23]:
percent_missing = wikipedia_movies.isnull().sum() * 100 / len(wikipedia_movies)
missing_value_df = pd.DataFrame(
    {"column_name": wikipedia_movies.columns, "percent_missing": percent_missing}
)

In [24]:
missing_value_df.sort_values("percent_missing", inplace=True)

In [25]:
missing_value_df

,column_name,percent_missing
year,year,0.000000
href,href,0.000000
title,title,0.000000
genres,genres,0.000000
omdb_response,omdb_response,0.000000
based_on_binary,based_on_binary,0.000000
based_on_binary_comic_books,based_on_binary_comic_books,0.000000
num_parentheticals,num_parentheticals,0.000000
rt_multi,rt_multi,0.000000
rt_binary,rt_binary,0.000000


In [26]:
movies_cleaned = movies_cleaned.rename(columns={"omdb_Title": "omdb_title"})

In [ ]:
wiki_tmdb_omdb_LOL = pd.merge(movies_cleaned, wikipedia_movies, on="omdb_title")

In [ ]:
wiki_tmdb_omdb_LOL_2 = movies_cleaned.merge(
    wikipedia_movies, on="omdb_title", how="left", indicator=True
)